# Healthcare Insurance Claims Analysis & Cost Optimization
**Dataset:** `Dataset/insurance.csv` (7 columns, one row per beneficiary record).

> **Scope note.** The dataset contains `age, sex, bmi, children, smoker, region, charges`. It has **no** patient ID, claim date,
> provider/insurer, diagnosis or admission type, so analyses that need those fields (monthly trends, provider comparison,
> condition/admission analysis, unique patients, fraud assessment) are **not performed**. Nothing is invented.

Run from the `Notebooks/` folder; the notebook imports the modules in `Python/`.

In [3]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "Python"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import analysis as an
import anomaly as ano
import cleaning as cl
from data_loader import load_raw

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")


ImportError: DLL load failed while importing _lsap: An Application Control policy has blocked this file.

In [2]:
%pip install -r ../requirements.txt


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: C:\Users\anura\AppData\Local\Python\pythoncore-3.15-64\python.exe -m pip install --upgrade pip


## A. Data understanding

In [ ]:
raw = load_raw()
print("Shape:", raw.shape)
raw.head()

In [ ]:
raw.dtypes.to_frame("dtype")

In [ ]:
raw.describe(include="all").T

### Missing values, duplicates and consistency checks

In [ ]:
qr = cl.quality_report(raw)
print("Missing cells:", qr["missing_total"])
print("Exact duplicate rows (beyond first occurrence):", qr["exact_duplicate_rows"])
print("Non-positive charges:", qr["charges_non_positive"], "| non-positive BMI:", qr["bmi_non_positive"])
print("Age range:", qr["age_range"], "| BMI range:", qr["bmi_range"], "| children range:", qr["children_range"])
print("sex:", qr["sex_values"], "| smoker:", qr["smoker_values"], "| region:", qr["region_values"])
qr["duplicate_rows_involved"]

**Finding:** no missing values, valid categorical labels, plausible ranges. One exact duplicate pair (all 7 fields identical, including `charges` to 4 decimals).
Because there is no ID column the pair cannot be proven to be double-entry, but identical charges to four decimals make a coincidence very unlikely; the second row is removed (effect on totals: 0.01 %).

## B. Data cleaning & feature engineering

In [ ]:
df = cl.clean(raw)
df = ano.add_peer_anomaly_flags(df)
print("Rows after cleaning:", len(df))
print("High-cost threshold (P90):", round(df.attrs["high_cost_threshold"], 2))
print("IQR upper fence:", round(df.attrs["iqr_upper_fence"], 2))
df.head()

Engineered columns: `age_group`, `bmi_category` (WHO), `has_children`, `smoker_flag`, `log_charges`, `charge_band`, `smoker_bmi_segment`,
`is_high_cost` (top 10 %), `is_iqr_outlier` (Tukey fence), peer anomaly fields. `record_id` is a surrogate row key, **not** a patient ID.

### Distribution and outliers

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df["charges"], bins=40, kde=True, ax=ax[0]); ax[0].set_title("Charges")
sns.histplot(df["log_charges"], bins=40, kde=True, ax=ax[1], color="grey"); ax[1].set_title("log(Charges)")
sns.boxplot(x=df["charges"], ax=ax[2]); ax[2].set_title("Charges boxplot")
plt.show()
print("Skewness of charges:", round(df["charges"].skew(), 2))
print("IQR outliers:", int(df["is_iqr_outlier"].sum()), "of", len(df))

## C. Exploratory data analysis

In [ ]:
k = an.kpis(df)
pd.Series(k).to_frame("value")

In [ ]:
by_age = an.group_summary(df, "age_group"); by_age

In [ ]:
by_sex = an.group_summary(df, "sex"); by_sex

In [ ]:
by_smoker = an.group_summary(df, "smoker"); by_smoker

In [ ]:
by_region = an.group_summary(df, "region"); by_region

In [ ]:
by_bmi = an.group_summary(df, "bmi_category"); by_bmi

In [ ]:
by_children = an.group_summary(df, "children"); by_children

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 8))
for ax, (tbl, col, t) in zip(axes.ravel(), [(by_age, "age_group", "Age group"), (by_sex, "sex", "Sex"),
        (by_smoker, "smoker", "Smoker"), (by_region, "region", "Region"),
        (by_bmi, "bmi_category", "BMI category"), (by_children, "children", "Children")]):
    sns.barplot(data=tbl, x=col, y="mean_charge", ax=ax, color="#2E86AB")
    ax.set_title(f"Average charge by {t}"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

### Relationships

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x="age", y="charges", hue="smoker", alpha=.6, ax=ax[0])
sns.scatterplot(data=df, x="bmi", y="charges", hue="smoker", alpha=.6, ax=ax[1]); ax[1].axvline(30, ls="--", c="grey")
plt.show()

In [ ]:
pearson, spearman = an.correlation_tables(df)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(pearson, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax[0]); ax[0].set_title("Pearson")
sns.heatmap(spearman, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax[1]); ax[1].set_title("Spearman")
plt.show()

Smoker status is by far the strongest linear correlate of charges; age is next. Spearman > Pearson for age shows the age relationship is monotonic but not linear in raw dollars.

## D. Advanced analysis

### D1. High-cost claims (top 10 % of charges)

In [ ]:
hc = df[df["is_high_cost"]]
print(f"High-cost records: {len(hc)} ({len(hc)/len(df):.1%} of records) -> {hc.charges.sum()/df.charges.sum():.1%} of total charges")
print("Smokers within high-cost:", f"{(hc.smoker=='yes').mean():.1%}", "| Obese within high-cost:", f"{(hc.bmi_category=='Obese').mean():.1%}")
an.group_summary(df, "smoker_bmi_segment")

In [ ]:
seg = df.pivot_table(index="smoker", columns="bmi_category", values="charges", aggfunc="mean", observed=True)
sns.heatmap(seg, annot=True, fmt=",.0f", cmap="YlOrRd"); plt.title("Average charge: smoker x BMI category"); plt.show()

### D2. Statistical testing (non-parametric because charges are right-skewed)

In [ ]:
an.statistical_tests(df)

Reading the table: smoker status has a very large effect; age group a large effect; BMI category and number of children small but significant; **sex and region are not significant** (p > 0.05).
Statistical significance here describes *association* in this sample, not causation.

### D3. Regression on log(charges) - adjusted associations

In [ ]:
model = an.log_linear_model(df)
print("R-squared (log scale):", round(model.attrs["r2"], 3), "| n =", model.attrs["n"])
model

`pct_effect` = approximate % change in charges for a one-unit change (or presence of the flag), holding the other fields fixed. The `smoker_x_obese` term shows that obesity is associated with higher charges mainly among smokers.

### D4. Anomaly detection vs. fraud
Three different ideas are kept separate on purpose:

| Concept | Question it answers | Rule |
|---|---|---|
| High-cost | Is this record expensive overall? | charges > P90 |
| Statistical outlier | Is it far from the whole distribution? | Tukey fence on all records |
| Peer anomaly | Is it unusual *compared with similar people*? | Tukey fence within smoker x age-group |

**None of these is evidence of fraud.** The data has no provider, date, diagnosis, procedure or labelled fraud cases.

In [ ]:
ano.anomaly_overlap(df)

In [ ]:
flag = df[df["is_peer_anomaly"]]
print("Peer anomalies:", len(flag), f"({len(flag)/len(df):.1%} of records, {flag.charges.sum()/df.charges.sum():.1%} of total charges)")
print("Of which smokers:", int((flag.smoker == 'yes').sum()), "| non-smokers:", int((flag.smoker == 'no').sum()))
flag[["record_id","age","sex","bmi","children","smoker","region","charges","peer_median_charge","charge_vs_peer_median"]] \
    .sort_values("charge_vs_peer_median", ascending=False).head(10)

In [ ]:
base = df[~df["is_peer_anomaly"]]
plt.figure(figsize=(9, 5))
plt.scatter(base.age, base.charges, s=12, alpha=.4, c="#9AA5B1", label="Within peer range")
plt.scatter(flag.age, flag.charges, s=35, c="#D1495B", label="Peer anomaly (review)")
plt.xlabel("Age"); plt.ylabel("Charges"); plt.legend(); plt.title("Peer-relative anomalies"); plt.show()

**Observation:** every peer anomaly is a *non-smoker* whose charges sit in a distinct mid-range band (mostly about 10K-37K, plus a few at roughly 4-5K among 18-24 year olds) well above the age-driven main band of non-smokers.
No field in this dataset explains that band, which makes it a data-collection question for the business (e.g. chronic conditions, procedures) rather than a conclusion.

### D5. Cost-optimisation opportunities (exposure, not savings)
The figures below size **where cost is concentrated**. They do not estimate achievable savings; that would require intervention-effect data this dataset does not contain.

In [ ]:
seg = an.group_summary(df, "smoker_bmi_segment")[["smoker_bmi_segment","records","pct_of_records","pct_of_total_charges","mean_charge"]]
seg

## Export check

In [ ]:
print(sorted(p.name for p in (Path.cwd().parent / "Dataset" / "processed").glob("*.csv")))

## Conclusions
Findings are written up with exact numbers in the project README; every number there comes from `Dataset/processed/*.csv` produced by `Python/run_pipeline.py` and verified by `Python/validate_outputs.py`.